In [1]:
import json
from tqdm.notebook import tqdm
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# !pip install pypdf langchain-text-splitters

In [7]:
def pdf_to_json_chunks(
    pdf_path: str,
    output_json_path: str,
    chunk_size: int = 1000, 
    chunk_overlap: int = 200,
) -> None:
    reader = PdfReader(pdf_path)
    full_text = ""
    
    for page_num, page in tqdm(enumerate(reader.pages),  total=len(reader.pages)):
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_text(full_text)

    json_data = []
    for index, chunk in tqdm(enumerate(chunks), total=len(chunks)):
        json_data.append({
            "chunk_id": index + 1,
            "text": chunk.strip(),
            "meta": {
                "character_count": len(chunk),
                "source_file": pdf_path
            }
        })

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)

pdf_file = "theoretical-texts/Antti_Ilmanen_Expected_Returns_A.pdf" 
output_json = "theoretical-texts/parsed_text.json"

pdf_to_json_chunks(
    pdf_path=pdf_file, 
    output_json_path=output_json, 
    chunk_size=1000,
    chunk_overlap=200,
)

  0%|          | 0/981 [00:00<?, ?it/s]

  0%|          | 0/1903 [00:00<?, ?it/s]